# Updating the variable metadata


This notebook takes in the cmor tables, as well as the ICON mapping and semi-automatically generates the pyku variable configuration. The result of this output must be included by hand and reviewed.

In [ ]:
import json
from itertools import islice

## Get official CMOR variables and metadata

In [ ]:
import pandas as pd
import yaml

url = "https://raw.githubusercontent.com/WCRP-CORDEX/data-request-table/refs/heads/main/cmor-table/datasets.csv"
df = pd.read_csv(url)

official_cmor_variables = {}

for _, row in df.iterrows():
    var_name = row['out_name']
    official_cmor_variables[var_name] = {
        'long_name': row['long_name'],
        'standard_name': row['standard_name'],
        'units': row['units'],
        'cmor_units': row['units'],
        'valid_bounds': [float(-1E20), float(1E20)] # Default
    }

# Print the first three entries
# -----------------------------

first_three_entries = dict(islice(official_cmor_variables.items(), 3))
print(json.dumps(first_three_entries, indent=4))

### Get ICON mapping to CMOR variables

In [ ]:
import re
import yaml
import requests

url = "https://gitlab.dkrz.de/udag/CORDEX-CMIP6_DaSt/-/raw/master/cmor/scripts/tables/ICON-CLM_EUR-12_CORDEX-CMIP6_mapping.txt"

response = requests.get(url)
lines = response.text.splitlines()

icon_variables = {}
pattern = re.compile(r'(\w+)=("[^"]*"|\S+)')

for line in lines:
    line = line.strip()
    if line.startswith('&parameter'):
        matches = pattern.findall(line)
        entry = {k: v.strip('"').strip('/') for k, v in matches}
        cmor_name = entry.get('cmor_name')
        name = entry.get('name')

        if name:
            icon_variables[cmor_name] = {
                'icon_name': entry.get('name', ''),
            }
            
first_three_entries = dict(islice(icon_variables.items(), 3))
print(json.dumps(first_three_entries, indent=4))

## Get the new variables only

In [ ]:
# Load existing variables
# -----------------------

with open('./pyku_dev/pyku/etc/variables.yaml', 'r') as f:
    existing_config = yaml.safe_load(f)
existing_variables = existing_config.get('variables', {}).keys()

# Filter new_vars to exclude existing ones
# ----------------------------------------

new_variables = {k: v for k, v in official_cmor_variables.items() if k not in existing_variables}

# Print first three entries
# -------------------------

first_three_entries = dict(islice(new_variables.items(), 3))
print(json.dumps(first_three_entries, indent=4))

### Add the ICON Mapping

In [ ]:
for var_name, details in new_variables.items():
    details['aliases'] = []
    if var_name in icon_variables:
        details['aliases'] = [icon_variables[var_name]['icon_name']]

first_three_entries = dict(islice(new_variables.items(), 3))
print(json.dumps(first_three_entries, indent=4))

### Print all entries for copy paste

In [ ]:
# def string_quoted_presenter(dumper, data):
#     # This forces single quote style for all scalar strings
#     return dumper.represent_scalar('tag:yaml.org,2002:str', data, style="'")

# yaml.add_representer(str, string_quoted_presenter)

print(yaml.dump(new_variables, sort_keys=False, default_flow_style=None))